PyCUDA : Embed C/C++ CUDA Code inside python Code

Numba : Python Library For Running Code Faster and allows GPU Code

CuPy : Numpy and Scipy For GPU

In [1]:
!pip install --no-cache-dir pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 45.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 355.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.3/102.3 kB 377.3 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp313-cp313-linux_x86_64.whl size=5316541 sha256=042e65e907d19c7800aa410b8a810a5e2d6c850a3efac0aef1c7e943510311c3
  Stored in directory: /tmp/pip-ephem-wheel-cache-0chruryn/wheels/ce/26/46/c519675fcb0e5e17bab8e85b6676528c40d12d794182340e85
Successfully built pycuda


In [7]:
%%writefile test.py

print("Hello From Python")
import pycuda.autoinit
import pycuda.driver as cuda

print(cuda.Device(0).name())

Writing test.py


In [8]:
!python test.py

Hello From Python
Tesla T4


In [14]:
%%writefile test.py
import pycuda.autoinit
import pycuda.driver as cuda
from pycuda.compiler import SourceModule
import numpy as np

mod = SourceModule("""
__global__ void add(int *a, int *b,int *c)
{
  int i = threadIdx.x;
  c[i] = a[i] + b[i];
}
""")

a = np.array([1,2,3,4,5], dtype=np.int32)
b = np.array([6,7,8,9,10], dtype=np.int32)
c = np.zeros(5, dtype=np.int32)

a_gpu = cuda.mem_alloc(a.nbytes)
b_gpu = cuda.mem_alloc(b.nbytes)
c_gpu = cuda.mem_alloc(c.nbytes)

cuda.memcpy_htod(a_gpu, a)
cuda.memcpy_htod(b_gpu, b)

func = mod.get_function("add")
func(a_gpu, b_gpu, c_gpu, block=(5,1,1))

cuda.memcpy_dtoh(c, c_gpu)

print(c)

Overwriting test.py


In [15]:
!python test.py

[ 7  9 11 13 15]
